In [1]:
# %% [markdown]
# Notebook ETL – leitura em batch, validações e write-back

# %% [code]

from pathlib import Path

# libs de terceiros
import pandas as pd
from rich.console import Console
from rich.logging import RichHandler
import logging, os, sys
from dotenv import load_dotenv               # se quiser .env

# console largo sem precisar de max_width
console = Console(width=120)
# ── logging bonito via rich ────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s › %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        RichHandler(
            console=console,
            rich_tracebacks=True,
            show_time=True,
            show_level=True,
            show_path=False,
            markup=True,          # permite [cyan]…[/] nos logs
        )
    ],
)
# ── variáveis de ambiente (opcional .env) ─────────────────
load_dotenv()

# Raiz do projeto no PYTHONPATH
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Logger para o notebook ────────────────────────────────────────────────
log = logging.getLogger(__name__)


In [2]:
import math
import numpy as np

def _to_json_safe(x):
    if x is None:
        return None
    if isinstance(x, (int, str, bool)):
        return x
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x) if not (math.isnan(x) or math.isinf(x)) else None
    if isinstance(x, (list, tuple, set)):
        return [_to_json_safe(i) for i in x]
    if isinstance(x, dict):
        return {k: _to_json_safe(v) for k, v in x.items()}
    return str(x)          # fallback: stringifica

def json_safe(obj):
    """Recursivamente converte obj em algo 100 % serializável para JSON."""
    return _to_json_safe(obj)


In [3]:
# %% [code]
# Flags de gravação (mude conforme necessidade)
WRITE_BACK_ORIGIN = True   # grava na aba-origem?        (meta*, tiktok*, …)
WRITE_BACK_DEST   = True    # grava nas abas-modelo?      (modelo*)
DRY_RUN_DEST      = False   # True = simula write-back destino

# Credenciais e planilha
CREDS_PATH     = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
SPREADSHEET_ID = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"

# Abas de origem a processar
SHEET_NAMES = [
    "metaGeral",   "metaIdade",   "metaGenero",   "metaRegiao",   "metaAlcance",
    "tiktokGeral", "tiktokIdade", "tiktokGenero", "tiktokRegiao", "tiktokAlcance",
    "pinterestGeral", "pinterestGenero", "pinterestIdade", "pinterestRegiao", "pinterestAlcance",
    "linkedinGeral", "linkedinRegiao", "linkedinAlcance",
]


In [4]:
# %% [code]
import importlib

# módulos principais
import extract.sheets_fetcher   as sf_mod
import treat.treat_pipeline     as tp_mod
import load.origin_writer       as ow_mod
import load.dest_writer         as dw_mod
from   treat.utils              import renomeacoes   as rn_mod
from   treat.utils.campos_calculados import (
    calcular_engajamento_total,
    gerar_id,
)

# hot-reload (útil quando editamos código no mesmo notebook)
for m in (sf_mod, tp_mod, ow_mod, dw_mod, rn_mod):
    importlib.reload(m)


In [5]:
# %% [code]
from typing import Dict
from contextlib import suppress
import gc
from safe_json import json_safe
from pprint    import pp
from extract.sheets_fetcher import SheetsFetcher
from treat.treat_pipeline   import TreatPipeline
from treat.utils.renomeacoes import (
    renomeacao_geral,
    renomear_colunas_origem_para_modelo,
)
from load.origin_writer import write_back_origin
from load.dest_writer   import write_back_for_sheet

# Instância única do fetcher (retry/backoff/cache interno)
fetcher = SheetsFetcher(
    spreadsheet_id = SPREADSHEET_ID,
    creds_path     = CREDS_PATH,
)
# %% [code]  ── Função utilitária de ETL por aba ─────────────────────────────
def run_etl_for_sheet(
    *,
    sheet: str,
    wb_origin_flag: bool,
    wb_dest_flag: bool,
    dry_run_dest: bool,
    preloaded_raw: pd.DataFrame,
) -> dict[str, pd.DataFrame | dict]:
    """Executa todo o fluxo para uma aba e devolve estágios de interesse."""
    # 1) raw já em memória
    df_raw = preloaded_raw

    # 2) tratamento
    pipeline = TreatPipeline(
        creds_path         = CREDS_PATH,
        spreadsheet_id     = SPREADSHEET_ID,
        sheet_name         = sheet,
        mapping_renomeacao = renomeacao_geral,
        write_back         = wb_origin_flag,
    )
    df_ok = pipeline.run(df_raw)
    # sanitiza e imprime o relatório de taxonomia sem np.nan ou tipos numpy
    taxo = json_safe(pipeline._last_taxo_report)
    pp(taxo, width=120)

    # relatório de taxonomia salvo pelo pipeline
    taxo_report = getattr(pipeline, "_last_taxo_report", {})

    # 3) write-back origem
    df_origin = write_back_origin(
        df_raw, df_ok,
        creds_path      = CREDS_PATH,
        spreadsheet_id  = SPREADSHEET_ID,
        sheet_name      = sheet,
        write_back      = wb_origin_flag,
        dry_run         = not wb_origin_flag,
    )
    if df_origin is None:
        df_origin = pd.DataFrame()

    # 4) modelagem
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 5) write-back destino
    df_dest = write_back_for_sheet(
        df_model,
        sheet_name     = sheet,
        creds_path     = CREDS_PATH,
        spreadsheet_id = SPREADSHEET_ID,
        write_back     = wb_dest_flag,
        dry_run        = dry_run_dest,
    )
    if df_dest is None:
        df_dest = pd.DataFrame()

    # devolve só o que interessa
    return {"dest": df_dest, "taxo": taxo_report}


In [6]:
from contextlib import suppress
import gc, pandas as pd, logging
from tqdm.auto import tqdm           # <-- barra de progresso
from load.dest_writer import prefetch_meta

# 1) batch único
all_raw = fetcher.get(SHEET_NAMES)
prefetch_meta(CREDS_PATH, SPREADSHEET_ID)
log.info("Abas carregadas: %s", list(all_raw))

# debug rápido
with suppress(KeyError):
    dbg = all_raw["linkedinRegiao"]
    log.debug("linkedinRegiao colunas=%s\n%s",
              dbg.columns.tolist(), dbg.head(2).T)

# 2) loop
results: dict[str, dict[str, object]] = {}

for sheet in tqdm(SHEET_NAMES, desc="Processando abas"):
    log.info("▶️  %s …", sheet)

    dfs = run_etl_for_sheet(
        sheet           = sheet,
        wb_origin_flag  = WRITE_BACK_ORIGIN,
        wb_dest_flag    = WRITE_BACK_DEST,
        dry_run_dest    = DRY_RUN_DEST,
        preloaded_raw   = all_raw[sheet],
    )

    # guarda apenas o que interessa
    results[sheet] = {"dest": dfs["dest"], "taxo": dfs["taxo"]}

    shapes = {k: (f"{v.shape[0]:,}×{v.shape[1]}" if isinstance(v, pd.DataFrame) else "—")
              for k, v in dfs.items()}
    log.info("Shapes: %s", shapes)

    dfs.clear(); gc.collect()


15:22:32 INFO     15:22:32 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ',          
                  'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ',       
                  'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ',                   
                  'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ',         
                  'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ']

15:22:35 INFO     15:22:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AQ8776

         INFO     15:22:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaIdade!A1:Q4282

         INFO     15:22:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGenero!A1:Q1730

         INFO     15:22:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaRegiao!A1:O20004

...

         INFO     15:25:47 INFO load.origin_writer › Write-back 'linkedinAlcance': 259 linhas, 7 colunas

         INFO     15:25:47 INFO load.origin_writer › ✅ Write-back concluído para 'linkedinAlcance'

/home/debrito/Documentos/etl_debrito/load/dest_writer.py:167: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(_scalar)


         INFO     15:25:47 INFO load.dest_writer › ✅ Gravadas 258 linha(s) em 'modeloAlcance'

         INFO     15:25:47 INFO __main__ › Shapes: {'dest': '258×13', 'taxo': '—'}

In [7]:
# %% [code]
# ── 1) leitura única (batchGet) de todas as abas ──────────
all_raw = fetcher.get(SHEET_NAMES)
log.info("[bold green]Abas carregadas:[/] %s", list(all_raw.keys()))

# ── debug opcional de uma aba específica ------------------
with suppress(KeyError):
    dbg = all_raw["linkedinRegiao"]
    log.debug("linkedinRegiao > colunas=%s\n%s", dbg.columns.tolist(), dbg.head(2).T)

# ── 2) processa cada aba em memória -----------------------
results: dict[str, dict[str, object]] = {}

for sheet in SHEET_NAMES:
    log.info("[cyan]▶️  Processando %s …[/]", sheet)

    dfs = run_etl_for_sheet(
        sheet          = sheet,
        wb_origin_flag = WRITE_BACK_ORIGIN,
        wb_dest_flag   = WRITE_BACK_DEST,
        dry_run_dest   = DRY_RUN_DEST,
        preloaded_raw  = all_raw[sheet],
    )

    # guarda somente destino + relatório de taxonomia para poupar RAM
    results[sheet] = {"dest": dfs["dest"], "taxo": dfs["taxo"]}

    # loga tamanhos resumidos
    shapes = {
        k: f"{v.shape[0]:,}×{v.shape[1]}" if isinstance(v, pd.DataFrame) else "-"
        for k, v in dfs.items()
    }
    log.info("[green]Shapes:[/] %s", shapes)

    # libera memória dos dfs grandes
    dfs.clear()
    gc.collect()


15:25:48 INFO     15:25:48 INFO extract.sheets_fetcher › 📥 Cache hit para ('linkedinAlcance', 'linkedinGeral',         
                  'linkedinRegiao', 'metaAlcance', 'metaGenero', 'metaGeral', 'metaIdade', 'metaRegiao',                
                  'pinterestAlcance', 'pinterestGenero', 'pinterestGeral', 'pinterestIdade', 'pinterestRegiao',         
                  'tiktokAlcance', 'tiktokGenero', 'tiktokGeral', 'tiktokIdade', 'tiktokRegiao')

         INFO     15:25:48 INFO __main__ › Abas carregadas: ['metaGeral', 'metaIdade', 'metaGenero', 'metaRegiao',      
                  'metaAlcance', 'tiktokGeral', 'tiktokIdade', 'tiktokGenero', 'tiktokRegiao', 'tiktokAlcance',         
                  'pinterestGeral', 'pinterestGenero', 'pinterestIdade', 'pinterestRegiao', 'pinterestAlcance',         
                  'linkedinGeral', 'linkedinRegiao', 'linkedinAlcance']

         INFO     15:25:48 INFO __main__ › ▶️  Processando metaGeral …

         WARNING  15:25:48 WARNING treat.utils.validations › [Validação] 42 valor(es) de 'ad_group_name' fora da        
                  BI_PARAMETRIZAÇÃO (taxonomy_ad_group_name):                                                           
                  ['2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_2025_CATALISA0001',                                      
                  '2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_2025_CATALISA0004',                                       
                  '2025_3_BR_CARD_CINTIA_ACAO_DBT_SBRAE_2025_EMP_FEM0013',                                              
                  '2025_3_BR_CARD_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM0028',                                                 
                  '2025_3_BR_CARD_SILVANA_ACAO_DBT_SBRAE_2025_EMP_FEM0044',                                             
                  '2025_3_BR_STORIES_CINTIA_ACAO_DBT_SBRAE_2025_EMP_FEM0021',                                           
                  '2025_3_BR_STORIES_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0061',                                        
                  '2025_3_BR_STORIES_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM0041',                                              
                  '2025_3_BR_STORIES_SILVANA_ACAO_DBT_SBRAE_2025_EMP_FEM0057',                                          
                  '2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0056',                                          
                  '2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0064',                                          
                  '2025_3_BR_CARD_CERRADO_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0150',                           
                  '2025_3_BR_CARD_CERRADO_DIA_DAS_INSCRIÇÕES_ACAO_DBT_SBRAE_2025_CER_PAN0146',                          
                  '2025_3_BR_CARD_CERRADO_MOTIVOS_PARA_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0152',                      
                  '2025_3_BR_CARD_CERRADO_PANTANAL_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0153',                  
                  '2025_3_BR_CARD_CERRADO_QUEM_PODE_PARTICIPAR_ACAO_DBT_SBRAE_2025_CER_PAN0142',                        
                  '2025_3_BR_CARD_CERRADO_QUEM_PODE_PARTICIPAR_ACAO_DBT_SBRAE_2025_CER_PAN0149',                        
                  '2025_3_BR_CARD_PANTANAL_CERRADO_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0144',                  
                  '2025_3_BR_CARD_PANTANAL_CERRADO_DIA_DAS_INSCRIÇÕES_ACAO_DBT_SBRAE_2025_PAN0140',                     
                  '2025_3_BR_CARD_PANTANAL_CERRADO_MOTIVOS_PARA_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0145'] …

         WARNING  15:25:48 WARNING treat.utils.validations › [Validação] 34 valor(es) de 'ad_name' fora da              
                  BI_PARAMETRIZAÇÃO (taxonomy_ad_name): ['2025_3_BR_CARD_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM0028',          
                  '2025_3_BR_CARD_SILVANA_ACAO_DBT_SBRAE_2025_EMP_FEM0044',                                             
                  '2025_3_BR_STORIES_CINTIA_ACAO_DBT_SBRAE_2025_EMP_FEM0021',                                           
                  '2025_3_BR_STORIES_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0061',                                        
                  '2025_3_BR_STORIES_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM0041',                                              
                  '2025_3_BR_STORIES_SILVANA_ACAO_DBT_SBRAE_2025_EMP_FEM0057',                                          
                  '2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0056',                                          
                  '2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0064',                                          
                  '2025_3_BR_CARD_CERRADO_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0150',                           
                  '2025_3_BR_CARD_CERRADO_DIA_DAS_INSCRIÇÕES_ACAO_DBT_SBRAE_2025_CER_PAN0146',                          
                  '2025_3_BR_CARD_CERRADO_MOTIVOS_PARA_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0152',                      
                  '2025_3_BR_CARD_CERRADO_PANTANAL_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0153',                  
                  '2025_3_BR_CARD_CERRADO_QUEM_PODE_PARTICIPAR_ACAO_DBT_SBRAE_2025_CER_PAN0142',                        
                  '2025_3_BR_CARD_CERRADO_QUEM_PODE_PARTICIPAR_ACAO_DBT_SBRAE_2025_CER_PAN0149',                        
                  '2025_3_BR_CARD_PANTANAL_CERRADO_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0144',                  
                  '2025_3_BR_CARD_PANTANAL_CERRADO_DIA_DAS_INSCRIÇÕES_ACAO_DBT_SBRAE_2025_PAN0140',                     
                  '2025_3_BR_CARD_PANTANAL_CERRADO_MOTIVOS_PARA_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0145',             
                  '2025_3_BR_VÍDEO_PANTANAL_CERRADO_BABI_ACAO_DBT_SBRAE_2025_CER_PAN0155',                              
                  '2025_3_BR_VÍDEO_ANA_LUISA_ACAO_DBT_SBRAE_2025_CER_PAN0177',                                          
                  '2025_3_BR_VÍDEO_CARTAS_DANIELLE_ACAO_DBT_SBRAE_2025_EMP_FEM0161'] …

...

         INFO     15:27:32 INFO load.origin_writer › Write-back 'linkedinAlcance': 259 linhas, 7 colunas

         INFO     15:27:32 INFO load.origin_writer › ✅ Write-back concluído para 'linkedinAlcance'

/home/debrito/Documentos/etl_debrito/load/dest_writer.py:167: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(_scalar)


         INFO     15:27:32 INFO load.dest_writer › Destino 'modeloAlcance': nenhuma linha nova para gravar

         INFO     15:27:32 INFO __main__ › Shapes: {'dest': '0×0', 'taxo': '-'}

In [8]:
from extract.sheets_fetcher  import SheetsFetcher
from treat.bi_param_utils   import BIParamLookup

fetcher = SheetsFetcher(SPREADSHEET_ID, CREDS_PATH)
# limpa cache de leituras em lote
fetcher.refresh(SHEET_NAMES)
# limpa cache de BI_PARAMETRIZAÇÃO em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

def df(self) -> pd.DataFrame:
    self._ensure_df()
    df = BIParamLookup._df.copy()
    # garanta que a coluna de “norm” existe:
    if "taxonomy_campaign_name_norm" not in df.columns:
        df["taxonomy_campaign_name_norm"] = (
            df["taxonomy_campaign_name"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    return df


         INFO     15:27:32 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ',          
                  'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ',       
                  'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ',                   
                  'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ',         
                  'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ']

15:27:35 INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AQ8776

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaIdade!A1:Q4282

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGenero!A1:Q1730

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaRegiao!A1:O20004

...

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: pinterestAlcance!A1:Z1424

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: linkedinGeral!A1:V558

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: linkedinRegiao!A1:X14714

         INFO     15:27:35 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: linkedinAlcance!A1:Q980

         INFO     15:27:35 INFO extract.sheets_fetcher › 📡 batchGet 18 ranges

In [9]:
# em qualquer lugar, antes de chamar .df() novamente:
from treat.bi_param_utils import BIParamLookup
BIParamLookup._df = None
BIParamLookup._last_load = 0.0


In [10]:
from treat.utils.validations import validate_consistent_dates_across_models

# Teste manual com dois casos
df_modelo_geral = pd.DataFrame({
    "Campanha": ["Teste Consistente", "Teste Consistente", "Teste Inconsistente"],
    "Veiculo": ["LinkedIn", "LinkedIn", "LinkedIn"],
    "start":   ["2024-01-01", "2024-01-01", "2024-01-01"],
    "end":     ["2024-01-31", "2024-01-31", "2024-01-31"],
})

df_modelo_idade = pd.DataFrame({
    "Campanha": ["Teste Consistente", "Teste Consistente", "Teste Inconsistente"],
    "Veiculo": ["LinkedIn", "LinkedIn", "LinkedIn"],
    "start":   ["2024-01-01", "2024-01-01", "2099-12-31"],  # ← divergência aqui
    "end":     ["2024-01-31", "2024-01-31", "2024-01-31"],
})

dest_dfs_fake = {
    "modeloGeral": df_modelo_geral,
    "modeloIdade": df_modelo_idade,
}

issues = validate_consistent_dates_across_models(dest_dfs_fake)
if not issues.empty:
    display(issues)
else:
    print("✅ Nenhuma divergência detectada (esperado: falso negativo)")


,local,tipo,aba,Campanha,Veiculo,valores
0,entre-abas,start,"modeloGeral,modeloIdade",Teste Inconsistente,LinkedIn,"[2024-01-01, 2099-12-31]"


In [ ]:
from treat.utils.validations import validate_consistent_dates_across_models
import logging

logging.basicConfig(level=logging.INFO)  # ou WARNING

dest_dfs = {s: info["dest"] for s, info in results.items()}

inconsistências = validate_consistent_dates_across_models(dest_dfs)

if inconsistências is not None and not getattr(inconsistências, "empty", False):
    display(inconsistências)
else:
    print("✅ Nenhuma divergência de start/end entre modelos.")


KeyError: 'aba'